# React — Loading and errors

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> Topic 15 gave you the four states — loading, error, empty, success — and you have been
> rendering them ever since. This topic is about what you actually put on the screen for two of
> them, and about the failures those two states do not cover.

## LESSON 74 — Loading UI

A loading state is not a technical detail the user tolerates; it is a screen you designed, and
it is on screen for the moments they are paying most attention.

### Spinner or skeleton

A **spinner** says "something is happening". A **skeleton** says "something is happening, and
here is the shape of what is coming". The difference is not decorative — it is measurable.

Playground experiment 31 loads the same four rows twice, once behind a spinner and once behind a
skeleton, and measures how far the paragraph *below* the content moves when the rows arrive:

| loading UI | the content below moves |
|---|---|
| spinner | **84 px** |
| skeleton | **0 px** |

That jump is **layout shift**. The spinner occupied one line; the rows occupy five, so everything
underneath was pushed down at the moment the data landed. If the user had started reading — or
reaching for a button — the page moved under them.

A skeleton avoids it by taking up the space the real content will take. That is the whole trick,
and it is why a skeleton that is not the size of the content is just a grey spinner.

### When each is right

| situation | use |
|---|---|
| content with a known shape — a list, a card, a table | a skeleton |
| an unknown or tiny amount of content | a spinner |
| a button waiting on a submit | in-place text on the button, and disable it |
| a background refresh of data already on screen | keep the old data; a subtle indicator, if anything |

That last row matters more than it looks. Replacing rendered content with a spinner during a
refresh throws away something the user was reading in order to show them nothing. Keep the stale
content and mark it.

### The flash

A loading state that appears for 80 ms is worse than none: the user sees a flicker and cannot
read it. The fix is a **delay before showing the indicator** — if the data arrives first, no
indicator ever appears.

```jsx
// in your project — the shape, not a library
const [showSpinner, setShowSpinner] = useState(false);

useEffect(() => {
  if (!isLoading) return setShowSpinner(false);
  const id = setTimeout(() => setShowSpinner(true), 200);
  return () => clearTimeout(id);
}, [isLoading]);
```

That is LESSON 41's cleanup and LESSON 42's timer, doing a different job. The example cell works
out what such a delay does to a range of response times.

### Key Notes

- A spinner announces waiting; a skeleton reserves the space, so nothing jumps. Measured: 84 px
  against 0.
- Match the skeleton to the real content's size, or it is only a grey spinner.
- On a refresh, keep the data that is already on screen.
- Delay the indicator ~200 ms so fast responses never flash one.

### Example

**Runnable — plain JS.** What a 200 ms delay actually buys, over a spread of response times. No
React here — this is arithmetic about timing, and it applies to any UI you ever build.

In [ ]:
// L74 — should the spinner have appeared at all?

const l74Delay = 200;                        // wait this long before showing anything
const l74MinVisible = 400;                   // and if shown, keep it at least this long

function l74Plan(responseMs) {
  if (responseMs <= l74Delay) {
    return { shown: false, visibleFor: 0, note: "no indicator at all" };
  }
  const naturalVisible = responseMs - l74Delay;
  const visibleFor = Math.max(naturalVisible, l74MinVisible);
  return {
    shown: true,
    visibleFor,
    note: visibleFor > naturalVisible ? `held ${visibleFor - naturalVisible}ms longer to avoid a flash` : "",
  };
}

console.log("response   indicator   visible for");
for (const ms of [50, 120, 200, 260, 400, 900, 3000]) {
  const plan = l74Plan(ms);
  console.log(
    `${String(ms + "ms").padEnd(10)} ${(plan.shown ? "yes" : "no").padEnd(11)} ${String(plan.visibleFor + "ms").padEnd(8)} ${plan.note}`,
  );
}

// the trade the delay makes, stated as numbers
const l74Samples = [50, 120, 200, 260, 400, 900, 3000];
const l74Suppressed = l74Samples.filter((ms) => !l74Plan(ms).shown).length;
console.log(
  `\n${l74Suppressed} of ${l74Samples.length} responses never show an indicator at all;`,
  `the slowest response waits ${l74Delay}ms before the user sees anything happening.`,
);

### Exercise

Parts 1 and 2 are **runnable — plain JS**; part 3 is **in the playground**.

1. The 200 ms delay has a cost: on a slow response the user stares at an unchanged screen for
   200 ms before anything acknowledges their click. Write `l74Compare(delays, responses)` that,
   for delays of 0, 100, 200 and 500 ms, reports how many of a set of response times would show
   an indicator and what the longest silent wait would be. Which delay would you ship, and what
   did you trade?
2. Add a minimum-visible rule that is *conditional*: only hold the indicator open if it has been
   visible less than 200 ms. Explain in a comment why holding a spinner open longer than
   necessary can be the right thing to do.
3. **In the playground**, in `31-loading-ui.jsx`: make the skeleton wrong on purpose — three
   rows instead of four — and measure the shift again with `window.__shift`. Then make it right
   and check you are back to zero.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**In your project**, with no cell — this is a design review of Mini-project 3.

Open it and answer for each of its loading states: is it a spinner or a skeleton, does the
content below move when the data arrives, and does anything flash on a fast response?

Then pick the worst one and fix it. One is enough.

Finally, in writing: your search box refreshes results as the user types (LESSON 42). During that
refresh, should the old results stay on screen or be replaced by a loading state? Argue it in two
sentences, and say what you would show instead of either.

## LESSON 75 — Error boundaries

Every error you have handled so far was one you expected: a failed request, an invalid form. This
lesson is about the other kind — a component throws while rendering, and without a boundary the
**entire app** unmounts and the user is left with a blank page.

### The minimal implementation

This is React's own, and it is the only class component in this course:

```jsx
class ErrorBoundary extends React.Component {
  constructor(props) {
    super(props);
    this.state = { hasError: false };
  }

  static getDerivedStateFromError(error) {
    // Update state so the next render will show the fallback UI.
    return { hasError: true };
  }

  componentDidCatch(error, info) {
    logErrorToMyService(error, info.componentStack);
  }

  render() {
    if (this.state.hasError) {
      return this.props.fallback;
    }
    return this.props.children;
  }
}
```

```jsx
<ErrorBoundary fallback={<p>Something went wrong</p>}>
  <Profile />
</ErrorBoundary>
```

Two methods, two jobs. `getDerivedStateFromError` runs **during render** and may only return new
state — so it decides *what to show*. `componentDidCatch` runs **after the commit** and may have
side effects — so it is where *logging* goes, with `info.componentStack` telling you where in
your tree it happened.

> **Yes, it is a class.** React is explicit: *"There is currently no way to write an Error
> Boundary as a function component."* This is not a way into class components — you will copy
> this thirty-line file into a project once and never think about it again. Everything you know
> about function components still applies to everything inside it.

### What it does not catch

The list is short, and the first item is the one that catches people out. Error boundaries do not
catch errors in:

- **event handlers**
- **asynchronous code** such as a `setTimeout` callback
- **server-side rendering**
- **the boundary itself**, as opposed to its children

Experiment 32 has a button for each of the first two, and a counter that proves the boundary
never heard about them. The reason is not a React limitation — it is how `try`/`catch` works in
JavaScript, which the example cell shows in six lines. React can only catch what happens inside
the call it makes to your component: by the time your click handler runs, that call returned long
ago.

So a `fetch` that rejects inside an Effect is **not** an error-boundary case. That is topic 15's
error state, and you already handle it.

The documented exception, worth knowing because it is easy to miss: errors thrown inside the
function passed to `startTransition` (LESSON 72) *are* caught by error boundaries.

### Where to put them

Not one at the root. A boundary around the whole app turns any failure into a whole-page error —
which is what you were trying to avoid.

Put them around **sections that can fail independently**: a widget, a panel, a route. Experiment
32 puts one around a section and leaves a paragraph outside it; when the section throws, the
paragraph stays. That is the goal — the rest of the app keeps working.

A root boundary is still worth having as the last line of defence, underneath the section-level
ones.

### Recovering

The boundary's fallback can offer a retry, which resets its own state and re-renders the
children. Measured in experiment 32: press "try again" while the cause is still there and the
boundary catches again immediately, straight back to the fallback. Retrying is only useful when
something about the cause has changed — new props, a new key, a fixed input. LESSON 76 is about
doing that honestly.

React also prints its own message in the console when a boundary catches, naming the component
and what it will do:

```text
The above error occurred in the <Risky> component. React will try to recreate this component
tree from scratch using the error boundary you provided, ErrorBoundary.
```

### Key Notes

- A render error with no boundary unmounts the whole app. A boundary replaces one subtree.
- `getDerivedStateFromError` chooses the fallback; `componentDidCatch` logs. It must be a class.
- It does **not** catch event handlers, async callbacks, SSR, or its own errors — expected
  failures still use topic 15's error state.
- Wrap sections that can fail independently, not just the root.

### Example

**Runnable — plain JS.** Why the "does not catch" list is what it is. Nothing React-specific
here: it is the behaviour of `try`/`catch` that decides what a boundary can see.

In [ ]:
// L75 — what a try/catch can and cannot catch

function l75Catching(label, run) {
  try {
    run();
    console.log(`${label.padEnd(28)} no error reached the catch block`);
  } catch (error) {
    console.log(`${label.padEnd(28)} CAUGHT: ${error.message}`);
  }
}

// 1. a synchronous throw — this is the render case
l75Catching("synchronous throw", () => {
  throw new Error("during the call");
});

// 2. a throw from a callback that runs later — this is the event-handler case
const l75Handlers = [];
l75Catching("registering a handler", () => {
  l75Handlers.push(() => {
    throw new Error("in a handler, long after the call returned");
  });
});

// the call above returned cleanly. now something else invokes the handler:
try {
  l75Handlers[0]();
} catch (error) {
  console.log("only a try/catch AROUND THE CALL sees it:", error.message);
}

// 3. a throw inside setTimeout — nothing can catch it from out here
l75Catching("scheduling a timeout", () => {
  setTimeout(() => {
    console.log("(the timeout callback ran; a throw here reaches nobody)");
  }, 0);
});

// 4. a rejected promise is not a throw at all
l75Catching("starting a rejecting promise", () => {
  Promise.reject(new Error("rejected")).catch((e) =>
    console.log("only .catch (or await + try) sees:", e.message),
  );
});

console.log(
  "\nReact calls your component and wraps THAT call. Anything that happens after the call",
  "returns — a click, a timer, a settled promise — is outside the wrapper, which is exactly",
  "the documented list.",
);

### Exercise

**In the playground**, in `32-error-boundary.jsx`.

1. Press each of the three throw buttons and record, for each, whether the fallback appeared and
   whether `window.__boundaryCaught` changed. Then explain the handler case in one sentence using
   the example cell above.
2. `componentDidCatch` logs twice for a single render error. Explain why, using what you know
   about the playground's `main.jsx` (LESSON 3), and say whether it would happen in production.
3. Move the `<p id="outside">` paragraph *inside* the boundary and throw during render again.
   Describe what the user now loses, and state the rule about boundary placement in your own
   words.
4. Give the boundary a `fallback` **prop** instead of hard-coding its fallback UI, so two
   different sections can show different messages. Then answer: why is a `fallback` prop a better
   design than a `message` string prop? (LESSON 67, if you need the argument.)

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** A triage table, because the most common mistake with boundaries is
reaching for one when the answer is an ordinary error state.

Write `l75Triage(failure)` returning `"error boundary"`, `"error state (topic 15)"` or `"neither
— fix the code"` for each of these, plus a one-line reason:

```
1. the API returns 500 for a request in an Effect
2. a component reads `user.name` and `user` is undefined because the prop was never passed
3. a click handler calls JSON.parse on invalid text
4. the user submits a form with an empty required field
5. a third-party chart component throws while rendering on some data shapes
6. the network is offline
```

Then answer in a comment: two of these six would show the user a blank page in an app with no
boundary at all. Which two, and what do they have in common that the others do not?

In [ ]:
// Your code here

## LESSON 76 — Retry, fallback data, and what you tell the user

You can now show a failure. This lesson is about what happens next, and it is mostly about
judgement rather than API.

### Not all failures are the same

| what happened | can retrying help? | what the user should see |
|---|---|---|
| network dropped, timeout, 502, 503 | **yes** | "Couldn't reach the server" + a retry |
| 500 from the server | sometimes | an apology + a retry |
| 404 — this thing does not exist | no | "Not found" — an empty state, not an error (L46) |
| 401 / 403 — not allowed | no | sign in, or "you don't have access" |
| 400 — the request was wrong | no | a message about the input, next to the field |

The single most useful question: **would doing exactly the same thing again plausibly work?**
Only that group gets a retry button. Offering "try again" for a 403 is a button that is
guaranteed not to help, and the user will press it three times before believing you.

### Retrying properly

Two rules, both about not making things worse.

**Back off.** If the server is struggling, five immediate retries from every client is the worst
possible response. Wait longer each time — 300 ms, 600 ms, 1200 ms — and stop after a few
attempts. The example cell implements this; it is ordinary JavaScript you can lift into any
project.

**Retry only what is retryable.** Wrap the decision in a function, so the rule lives in one place
next to the service layer from LESSON 48:

```js
function isRetryable(status) {
  return status === 0 || status === 408 || status === 429 || status >= 500;
}
```

A manual retry button, by the way, is often better than any automatic policy: it costs the user
one click, it never hammers the server, and it happens when they are actually looking at the
screen.

### Fallback data

Sometimes you can show something rather than nothing: the previous successful response, a cached
copy, a sensible default. This is worth doing when stale data is genuinely useful and clearly
wrong-able — a dashboard, a list of articles.

It is worth **not** doing when stale data could be mistaken for current and acted on: a balance,
a stock level, a "who is on call right now". If you show it, say so — *"showing data from 14:02;
couldn't refresh"* — and never show stale data as though it were fresh.

### Two different audiences

Every failure produces two pieces of writing, and confusing them is the classic mistake:

| | the user | the log |
|---|---|---|
| contains | what happened, what they can do | the stack, the URL, the status, the ids |
| tone | plain language, no blame | precise and complete |
| example | "Couldn't load your projects. Check your connection and try again." | `GET /api/projects 503 · reqId 8f21 · user 402` |

Never put the exception message on screen. `TypeError: Cannot read properties of undefined` tells
the user nothing, tells an attacker something, and tells you nothing either unless it also went
to your log. `componentDidCatch` (LESSON 75) is exactly the place for the second column.

Three things a good user-facing message has: it says what failed in their words ("your
projects"), it says whether they can do anything, and it does not blame them.

### Key Notes

- Retry only when repeating the request could plausibly work — 5xx and network, not 403 or 404.
- Back off between attempts and cap them. A manual retry button is often the better answer.
- Fallback data is fine when it is useful and labelled as stale; never for numbers people act on.
- Two audiences: a plain sentence for the user, the full detail for the log. Never swap them.

### Example

**Runnable — plain JS.** A retry with backoff, written properly, against a flaky mock. This is
real code for your services layer, not a demonstration.

In [ ]:
// L76 — retry with backoff

function l76IsRetryable(status) {
  return status === 0 || status === 408 || status === 429 || status >= 500;
}

const l76Wait = (ms) => new Promise((resolve) => setTimeout(resolve, ms));

async function l76WithRetry(request, { attempts = 3, baseDelay = 100 } = {}) {
  const log = [];
  for (let attempt = 1; attempt <= attempts; attempt += 1) {
    const result = await request(attempt);
    log.push(`attempt ${attempt}: status ${result.status}`);

    if (result.ok) return { ok: true, data: result.data, log };

    if (!l76IsRetryable(result.status)) {
      log.push(`  status ${result.status} is not retryable — stopping`);
      return { ok: false, status: result.status, log };
    }

    if (attempt < attempts) {
      const delay = baseDelay * 2 ** (attempt - 1);       // 100, 200, 400…
      log.push(`  waiting ${delay}ms before retrying`);
      await l76Wait(delay);
    }
  }
  return { ok: false, status: 0, log, gaveUp: true };
}

// a mock that fails twice with 503, then succeeds
function l76FlakyServer(attempt) {
  return Promise.resolve(
    attempt < 3
      ? { ok: false, status: 503 }
      : { ok: true, status: 200, data: { projects: ["Apollo", "Gemini"] } },
  );
}

const l76Recovered = await l76WithRetry(l76FlakyServer);
console.log("flaky server:");
for (const line of l76Recovered.log) console.log("  " + line);
console.log("  result:", l76Recovered.ok ? l76Recovered.data : `failed (${l76Recovered.status})`);

// and one that is simply forbidden
const l76Forbidden = await l76WithRetry(() => Promise.resolve({ ok: false, status: 403 }));
console.log("\nforbidden:");
for (const line of l76Forbidden.log) console.log("  " + line);
console.log("  attempts made:", l76Forbidden.log.filter((l) => l.startsWith("attempt")).length);

### Exercise

**Runnable — plain JS.**

1. `l76WithRetry` gives up after three attempts and returns `{ ok: false, status: 0 }`, which a
   caller cannot tell apart from a network failure on the first try. Fix the return value so the
   caller knows how many attempts were made and what the last status was, and write the two
   different user-facing sentences you would show for "gave up after 3 attempts" and "403".
2. Add **jitter**: instead of exactly 100/200/400 ms, wait a random amount between 50% and 100%
   of the computed delay. Then explain in a comment what problem jitter solves that plain backoff
   does not — think about a thousand clients whose requests all failed at the same instant.
3. Write `l76Message(failure)` that turns `{ status, resource }` into the sentence you would put
   on screen, for statuses 0, 403, 404, 429 and 500. No status codes, no exception text, and
   every sentence must say whether the user can do something.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Stale data, honestly.

Write `l76Cache` with `set(key, value)` and `get(key)` that also records **when** the value was
stored, and `l76Display(key, fresh)` that returns what to render: the fresh value if it arrived,
otherwise the cached value with its age and a note, otherwise the error state.

Then use it for two cases — a list of articles and an account balance — and print what each would
show after a failed refresh, 6 minutes after the last success.

Finally, answer in comments: your function returns the same shape for both, but one of them
should not use it at all. Which, and what is the rule you would write down for a colleague about
when stale data is allowed on screen?

In [ ]:
// Your code here

## LESSON 77 — `use`

LESSON 37 promised you this one. It named `use` as the API that breaks the rule every other Hook
follows, and pointed here.

> `use` is a React API that lets you read a resource during rendering, such as a Promise or
> context.

```jsx
const value = use(resource);
```

### It is not a Hook

React says so outright, and it is the reason the Rules of Hooks do not apply to it:

> **Despite its name, `use` is not a Hook.** Unlike Hooks, it can be called inside loops and
> conditional statements like `if`.

One rule remains: *"`use` must be called inside a Component or a Hook."* You cannot call it from
an event handler or a plain function.

### Reading a context

The simple half. `use(SomeContext)` does what `useContext(SomeContext)` does:

```jsx
function Button({ showTheme }) {
  let theme = null;
  if (showTheme) {
    theme = use(ThemeContext);      // legal — this is not a Hook
  }
  // …
}
```

Written with `useContext`, that `if` would be a Rules-of-Hooks violation. That is the entire
practical difference; where a conditional read is not needed, either will do.

### Reading a Promise

The interesting half:

> Call `use` with a Promise to read its resolved value. The component calling `use` *suspends*
> while the Promise is pending.

```jsx
function Greeting({ messagePromise }) {
  const message = use(messagePromise);      // suspends until it resolves
  return <p>{message}</p>;
}
```

Three outcomes, and each has a place that handles it:

| the Promise | what the user sees |
|---|---|
| pending | the nearest `<Suspense>` fallback |
| resolved | your component, with the value |
| rejected | the nearest **error boundary**'s fallback (LESSON 75) |

So the loading and error branches move *out* of the component. There is no `if (loading)` and no
`if (error)` — the component is written as though the data is simply there, and the two
boundaries around it handle the rest. Measured in experiment 33: click, the fallback appears,
then the greeting; ask for a failing one, and the error boundary shows *"the server said no"*.

This is the second legitimate use of Suspense in this course. The first was `React.lazy` for code
(LESSON 73).

### The rule that will bite you

> **Promises passed to `use` must be cached.** Promises created during render are recreated on
> every render, which causes React to show the Suspense fallback repeatedly and prevents content
> from appearing.

```jsx
function Albums() {
  const albums = use(fetch('/albums'));                 // 🔴 a new Promise every render
}
```

Experiment 33 has a button that does exactly this. Measured: the fallback is still on screen 2.5
seconds after a Promise that settles in 400 ms — it never finishes, because each render throws
away the one that was about to.

The same trap has quieter forms, all from React's own list: an uncached `async` function called
during render, and `use(fetchData(url).then(...))` — because `.then` returns a *new* Promise even
when `fetchData` is cached.

The fix is to create the Promise somewhere that is not render. Experiment 33 uses a module-level
`Map` keyed by the argument, which is why pressing "load Ada" a second time shows the greeting
with **no fallback at all** — the cached Promise is already resolved.

React's own advice about where Promises should come from:

> Ideally, Promises are created before rendering, such as in an event handler, a route loader, or
> a Server Component, and passed to the component that calls `use`.

### So should you use it for data fetching?

Not in this course, and be careful about it generally. Without a framework that manages the
cache for you, you have to build one — and a hand-rolled `Map` has no invalidation, no refetch,
no deduplication of in-flight requests, and no way to show a refresh.

For ordinary data loading, keep the explicit **loading / error / empty / success** states from
LESSON 46 and the Effect from LESSON 47. `use` earns its place when the Promise already exists —
created by a click, a route, or a framework — and you want the component to read it without
plumbing.

One more caveat, because the error is baffling the first time:

> You are calling `use` inside a try-catch block. `use` throws internally to integrate with
> Suspense, so it cannot be wrapped in try-catch.

Wrap the component in an error boundary instead.

### Key Notes

- `use` is not a Hook: it may be called conditionally. It must still be called inside a component
  or Hook.
- `use(Context)` reads context; `use(promise)` suspends until it resolves.
- Pending → the `<Suspense>` fallback. Rejected → the error boundary. No `if (loading)` in the
  component.
- The Promise **must** come from a cache. One created during render leaves the fallback on screen
  forever.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/33-use.jsx`.

Press **load Ada** and watch the fallback then the greeting. Press **load Grace**, then **load
Ada** again — no fallback the second time, because the cache returns the resolved Promise. Tick
the checkbox to read the theme context from inside an `if`. Press **load one that fails** and the
error boundary takes over. Then press the last button to watch the uncached-Promise mistake never
resolve.

The console logs each time a Promise is *created*, which is the number the cache exists to keep
low.

### Exercise

**In the playground**, in `33-use.jsx`.

1. Add a fourth button that loads `"Ada"` again but with a **different delay**. Does the cache
   return the existing Promise or make a new one, and is that the behaviour you want? Change the
   cache key if you disagree with what you find.
2. Remove the `key` from the `<ErrorBoundary>` and press: Ada, then the failing one, then Ada
   again. What happens, and what was the `key` doing? (LESSON 20's idea, in a place you did not
   expect it.)
3. Wrap the `use(...)` call in a `try`/`catch` and read the error React gives you. Quote it, then
   say what you must do instead.
4. Write the same `Greeting` component **without** `use` — `useState` + `useEffect` + the four
   states from LESSON 46 — and put the two versions side by side in a comment. Which is shorter,
   which would you rather debug, and what does the second one give you that the first does not?

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** The cache is the whole difficulty, so build one properly.

Write `l77Cache` with a `get(key, create)` that returns an existing Promise or creates and stores
one, counting hits and misses. Then use it to answer these, printing evidence for each:

1. Two "components" ask for the same key at the same time, before it resolves. How many Promises
   were created? (This is request deduplication, and it is free if the cache holds the Promise
   rather than the value.)
2. A request fails. What is now in the cache, and what happens the next time someone asks for
   that key? Fix it so a failure is not cached forever.
3. Add `invalidate(key)` and show a refetch working.

Then answer in a comment: you have just written the three features that make a data cache usable,
and you are nowhere near done — name two more that a real one needs. That list is the honest
answer to "why not use `use` for everything".

In [ ]:
// Your code here

> **Topic 24 complete — LESSON 74 to 77.** A loading state you designed rather than defaulted
> into, a boundary so one broken section cannot blank the page, a retry policy that knows which
> failures are worth retrying, and `use` — the API that lets a component read a resource and
> leaves loading and errors to the boundaries around it.
>
> Topic 25 is forms again, at full size: field arrays, validation as data, and React's Actions.